# DDPG/PID hybrid evaluation

Original comparison-controller definitions; evaluation launches and video-writing loops are omitted.

Selected source excerpts from the original research notebook. Outputs, execution counts, widget state and duplicate plotting trials are removed. Definitions are kept as code; configuration, training launches and hardware calls are displayed as reference text so Run All cannot start training or connect to a robot. This is a review extract, not a complete reproducibility package. See README.md for omitted dependencies.


In [ ]:
import os
import random
from collections import deque
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
device = torch.device("cpu")


## Source cell 3 (original zero-based index 2)


Reference only; not executed.

```python
import os
import torch
import numpy as np
from tqdm import tqdm
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
import time
from scipy.io import loadmat
import cv2
import gym
from collections import deque

# ------- BEGIN: User-provided / project-specific imports -------
# Make sure utils.MatlabEnv is on the PYTHONPATH or same folder
from utils import MatlabEnv_Static
from utils import MatlabEnv_Dynamic
# ------- END imports -------

# Configuration
current_dir = os.getcwd()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

# Path configurations (revised for Comparison subfolders)
PATHS = {
    'hybrid_best_weights': os.path.join(current_dir, 'Weights', 'Best_weights-1000 eps'),
    'hybrid_last_weights': os.path.join(current_dir, 'Weights', 'Last_weights-1000 eps'),
    'variables': os.path.join(current_dir, 'Variables-500 eps'),

    'renders_tuning': os.path.join(current_dir, 'Renders', 'Comparison', 'Tuning'),
    
    'renders_train_no': os.path.join(current_dir, 'Renders', 'Comparison', 'Train', 'no_disturbance'),
    'renders_train_feedback': os.path.join(current_dir, 'Renders', 'Comparison', 'Train', 'feedback_dist'),
    'renders_train_action_bias': os.path.join(current_dir, 'Renders', 'Comparison', 'Train', 'action_dist', 'bias'),
    'renders_train_action_impulse': os.path.join(current_dir, 'Renders', 'Comparison', 'Train', 'action_dist', 'impulse'),
    'renders_train_action_impulse_bias': os.path.join(current_dir, 'Renders', 'Comparison', 'Train', 'action_dist', 'impulse_bias'),
    'renders_train_action_sine': os.path.join(current_dir, 'Renders', 'Comparison', 'Train', 'action_dist', 'sine'),

    'renders_test_no': os.path.join(current_dir, 'Renders', 'Comparison', 'Test', 'no_disturbance'),
    'renders_test_feedback': os.path.join(current_dir, 'Renders', 'Comparison', 'Test', 'feedback_dist'),
    'renders_test_action_bias': os.path.join(current_dir, 'Renders', 'Comparison', 'Test', 'action_dist', 'bias'),
    'renders_test_action_impulse': os.path.join(current_dir, 'Renders', 'Comparison', 'Test', 'action_dist', 'impulse'),
    'renders_test_action_impulse_bias': os.path.join(current_dir, 'Renders', 'Comparison', 'Test', 'action_dist', 'impulse_bias'),
    'renders_test_action_sine': os.path.join(current_dir, 'Renders', 'Comparison', 'Test', 'action_dist', 'sine'),

    'plots_test_no': os.path.join(current_dir, 'Plots', 'Comparison', 'Test', 'no_disturbance'),
    'plots_test_feedback': os.path.join(current_dir, 'Plots', 'Comparison', 'Test', 'feedback_dist'),
    'plots_test_action_bias': os.path.join(current_dir, 'Plots', 'Comparison', 'Test', 'action_dist', 'bias'),
    'plots_test_action_impulse': os.path.join(current_dir, 'Plots', 'Comparison', 'Test', 'action_dist', 'impulse'),
    'plots_test_action_impulse_bias': os.path.join(current_dir, 'Plots', 'Comparison', 'Test', 'action_dist', 'impulse_bias'),
    'plots_test_action_sine': os.path.join(current_dir, 'Plots', 'Comparison', 'Test', 'action_dist', 'sine'),
    
    'matlab_workspaces': os.path.abspath(os.path.join(current_dir, '..', '..', '2) Matlab_Codes', '4) Workspaces', 'Cartesian'))
}

for path in PATHS.values():
    os.makedirs(path, exist_ok=True)
```


## Source cell 7 (original zero-based index 6)


In [ ]:
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim, action_max, action_min, hidden_sizes, state_min, state_max):
        super().__init__()
        self.register_buffer('action_max', torch.tensor(action_max, dtype=torch.float32))
        self.register_buffer('action_min', torch.tensor(action_min, dtype=torch.float32))
        self.register_buffer('state_min', torch.tensor(state_min, dtype=torch.float32))
        self.register_buffer('state_max', torch.tensor(state_max, dtype=torch.float32))
        layers = []
        input_dim = state_dim  # State only (3D position)
        for size in hidden_sizes:
            layers.extend([nn.Linear(input_dim, size), nn.LayerNorm(size), nn.ReLU()])
            input_dim = size
        layers.append(nn.Linear(input_dim, action_dim))
        self.net = nn.Sequential(*layers).to(device)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
            nn.init.zeros_(m.bias)
    
    def forward(self, state):
        state_norm = 2 * (state - self.state_min) / (self.state_max - self.state_min) - 1
        act_norm = torch.tanh(self.net(state_norm))
        act = 0.5 * (act_norm + 1) * (self.action_max - self.action_min) + self.action_min
        return act


## Source cell 10 (original zero-based index 9)


In [ ]:
class PID_Hybrid:
    def __init__(self, gains_xy, gains_z, dt=0.01):
        self.Kp_xy, self.Ki_xy, self.Kd_xy = gains_xy  # Gains for x and y (shared)
        self.Kp_z, self.Ki_z, self.Kd_z = gains_z      # Gains for z (separate)
        self.dt = dt                                   # Time step for continuous PID
        self.integral_xy = np.zeros(2, dtype=np.float32)  # Integral for x and y
        self.integral_z = 0.0                             # Scalar integral for z
        self.prev_error_xy = np.zeros(2, dtype=np.float32)  # Previous error for x and y
        self.prev_error_z = 0.0                           # Scalar previous error for z
    
    def compute(self, error):
        error = np.array(error, dtype=np.float32)
        error_xy = error[:2]  # Errors for x and y
        error_z = error[2]    # Error for z
        
        # Compute for x and y (shared gains, continuous)
        self.integral_xy += error_xy * self.dt  # Integral with dt
        derivative_xy = (error_xy - self.prev_error_xy) / self.dt if self.dt > 0 else np.zeros(2)  # Derivative with dt
        output_xy = (self.Kp_xy * error_xy + 
                     self.Ki_xy * self.integral_xy + 
                     self.Kd_xy * derivative_xy)
        
        # Compute for z (separate gains, continuous)
        self.integral_z += error_z * self.dt  # Integral with dt
        derivative_z = (error_z - self.prev_error_z) / self.dt if self.dt > 0 else 0.0  # Derivative with dt
        output_z = (self.Kp_z * error_z + 
                    self.Ki_z * self.integral_z + 
                    self.Kd_z * derivative_z)
        
        # Combine into 3D output (force correction; extend to 6D wrench if needed)
        output = np.concatenate([output_xy, [output_z]])
        
        self.prev_error_xy = error_xy
        self.prev_error_z = error_z
        return output
    
    def reset(self):
        self.integral_xy = np.zeros(2, dtype=np.float32)
        self.integral_z = 0.0
        self.prev_error_xy = np.zeros(2, dtype=np.float32)
        self.prev_error_z = 0.0


## Source cell 12 (original zero-based index 11)


In [ ]:
class PID_Standalone:
    def __init__(self, gains_force_xy, gains_force_z, gains_torque_xy, gains_torque_z, dt=0.01):
        """
        Initialize PID controller for 6D wrench output (Fx, Fy, Fz, Tx, Ty, Tz)
        
        Based on Cosserat rod theory:
        - Position error used to estimate required curvature (observability)
        - Torque commands derived from position error through static equilibrium
        - Separate gains for planar (xy) and vertical (z) directions
        - Continuous PID with time step dt
        """
        # Force gains (same as before)
        self.Kp_xy, self.Ki_xy, self.Kd_xy = gains_force_xy
        self.Kp_z, self.Ki_z, self.Kd_z = gains_force_z
        
        # Torque gains (NEW - derived from position error)
        self.Kp_txy, self.Ki_txy, self.Kd_txy = gains_torque_xy
        self.Kp_tz, self.Ki_tz, self.Kd_tz = gains_torque_z

        self.dt = dt  # Time step for continuous PID

        # Force state variables
        self.integral_xy = np.zeros(2, dtype=np.float32)
        self.integral_z = 0.0
        self.prev_error_xy = np.zeros(2, dtype=np.float32)
        self.prev_error_z = 0.0
        
        # Torque state variables (NEW)
        self.integral_torque_xy = np.zeros(2, dtype=np.float32)
        self.integral_torque_z = 0.0
        self.prev_torque_error_xy = np.zeros(2, dtype=np.float32)
        self.prev_torque_error_z = 0.0

    def compute(self, error, out_max, out_min):
        """Compute 6D wrench from 3D position error using Cosserat observability"""
        error = np.array(error, dtype=np.float32)
        error_xy = error[:2]  # Errors for x and y
        error_z = error[2]    # Error for z

        # FORCE CALCULATIONS (continuous)
        # Compute for x and y (shared gains)
        self.integral_xy += error_xy * self.dt
        derivative_xy = (error_xy - self.prev_error_xy) / self.dt if self.dt > 0 else np.zeros(2)
        output_force_xy = (self.Kp_xy * error_xy +
                           self.Ki_xy * self.integral_xy +
                           self.Kd_xy * derivative_xy)

        # Compute for z (separate gains)
        self.integral_z += error_z * self.dt
        derivative_z = (error_z - self.prev_error_z) / self.dt if self.dt > 0 else 0.0
        output_force_z = (self.Kp_z * error_z +
                          self.Ki_z * self.integral_z +
                          self.Kd_z * derivative_z)

        # TORQUE CALCULATIONS (continuous)
        # Based on Cosserat static equilibrium:
        # dM/ds + dR/ds × F = 0 → Torque relates to position curvature
        # Position error used as proxy for required curvature adjustment
        
        # Compute torque for xy plane (Tx, Ty)
        self.integral_torque_xy += error_xy * self.dt
        derivative_torque_xy = (error_xy - self.prev_torque_error_xy) / self.dt if self.dt > 0 else np.zeros(2)
        output_torque_xy = (self.Kp_txy * error_xy +
                            self.Ki_txy * self.integral_torque_xy +
                            self.Kd_txy * derivative_torque_xy)

        # Compute torque for z axis (Tz)
        self.integral_torque_z += error_z * self.dt
        derivative_torque_z = (error_z - self.prev_torque_error_z) / self.dt if self.dt > 0 else 0.0
        output_torque_z = (self.Kp_tz * error_z +
                           self.Ki_tz * self.integral_torque_z +
                           self.Kd_tz * derivative_torque_z)

        # Combine into 6D wrench [Fx, Fy, Fz, Tx, Ty, Tz]
        output = np.array([
            output_force_xy[0], output_force_xy[1], output_force_z,
            output_torque_xy[0], output_torque_xy[1], output_torque_z
        ])
        # Update force state
        self.prev_error_xy = error_xy
        self.prev_error_z = error_z
        
        # Update torque state
        self.prev_torque_error_xy = error_xy
        self.prev_torque_error_z = error_z
        return output
    
    def reset(self):
        """Reset all PID state variables"""
        # Force state
        self.integral_xy = np.zeros(2, dtype=np.float32)
        self.integral_z = 0.0
        self.prev_error_xy = np.zeros(2, dtype=np.float32)
        self.prev_error_z = 0.0
        
        # Torque state
        self.integral_torque_xy = np.zeros(2, dtype=np.float32)
        self.integral_torque_z = 0.0
        self.prev_torque_error_xy = np.zeros(2, dtype=np.float32)
        self.prev_torque_error_z = 0.0


## Source cell 14 (original zero-based index 13)


Reference only; not executed.

```python
# Simplified DDPG for inference (no buffer, no training)
class DDPGInference:
    def __init__(self, state_dim, action_dim, actor_hidden, state_min, state_max, act_high, act_low):
        # Actor
        self.actor = Actor(state_dim, action_dim, act_high, act_low, 
                           actor_hidden, state_min, state_max).to(device)
        
        # Bounds
        self.act_low = act_low
        self.act_high = act_high
        
        # Noise sigma based on action bounds
        self.action_sigma = np.minimum(np.abs(act_low) / 3, act_high / 3)

    def act(self, state, noise_level=0.0):
        with torch.no_grad():
            state_t = torch.as_tensor(state, device=device, dtype=torch.float32).unsqueeze(0)
            action = self.actor(state_t).cpu().numpy()[0]
            
            # Add noise if level > 0 (handled externally in render)
            if noise_level > 0:
                action += noise_level * self.action_sigma * np.random.randn(*action.shape)
                action = np.clip(action, self.act_low, self.act_high)
            
            return action

# Environment specs
matter = 'PDMS2'
mag_set = 'Set2'
env = MatlabEnv_Dynamic(matter=matter, mag_set=mag_set)

state_dim = env.observation_space.shape[0]  # 3 (position)
action_dim = env.action_space.shape[0]  # 6

low = env.observation_space.low
high = env.observation_space.high
state_min = low
state_max = high
act_low = env.action_space.low
act_high = env.action_space.high

# Hyperparams
actor_hidden = (400, 300)

# Init inference agent
agent = DDPGInference(state_dim, action_dim, actor_hidden, 
                      state_min, state_max, act_high, act_low)

# Load weights
actor_path = os.path.join(PATHS['hybrid_last_weights'], 'actor_dict.pth')

agent.actor.load_state_dict(torch.load(actor_path, map_location=device))
print("Actor model successfully loaded for Hybrid inference.")
```


## Source cell 17 (original zero-based index 16)


In [ ]:
from IPython.display import Video, display

class ComparisonSimulator:
    def __init__(self, env, agent, pid_hybrid, pid_standalone, seed=42):
        self.env = env
        self.agent = agent
        self.pid_hybrid = pid_hybrid
        self.pid_standalone = pid_standalone
        self.seed = seed
        self.low = env.action_space.low
        self.high = env.action_space.high
        self.pos_sigma_cache = None  # To cache if needed, but computed per run

    def _run_common_setup(self, file_path, cont_dt=0.1, sim_dt=0.01, velocity=1.0):
        datas = loadmat(file_path)
        targets = datas['datas']['Pos'][0, 0]  # 3D targets
        num_substeps = int(cont_dt / sim_dt)
        num_targets = len(targets)
        if velocity < 1.0:
            step_indices = np.linspace(0, num_targets - 1, int(num_targets * velocity), dtype=int)
            targets = targets[step_indices]
        pos_dist = np.ptp(targets, axis=0)[:3]
        pos_sigma = np.abs(pos_dist / 3)
        action_sigma = np.maximum(np.abs(self.low) / 3, self.high / 3)
        self.pos_sigma_cache = pos_sigma
        return targets, num_substeps, pos_sigma, action_sigma

    def run_drl(self, file_path, feedback_noise_level=0.0, action_impulse_level=0.0, 
                action_bias_level=0.0, bias_start_step=120, action_sine_level=0.0, 
                cont_dt=0.1, sim_dt=0.01, velocity=1.0):
        np.random.seed(self.seed)
        targets, num_substeps, pos_sigma, action_sigma = self._run_common_setup(file_path, cont_dt, sim_dt, velocity)
        
        # Reset for main simulation
        achieved_pos = self.env.reset(new_ep=1)
        achieved_pos = np.array([0.0, 0.0, self.env.observation_space.high[0]])
        action = np.zeros(6, dtype=np.float32)
        bias_offset = action_bias_level * np.random.uniform(low=self.low, high=self.high, size=action.shape)
        
        # Lists to record
        actual_positions = []
        target_positions = []
        
        previous_action = None
        
        # Main simulation loop
        for step in range(len(targets)):
            target = targets[step]
            
            # For pure DRL, input is directly the target (open loop)
            input_state = target
            
            # Get action from agent (deterministic, noise added separately)
            action = self.agent.act(input_state, noise_level=0.0)  # No noise here
            
            # Clip to bounds
            action = np.clip(action, self.low, self.high)
            
            # Compute delta_vec for impulse
            if step > 0:
                delta_vec = action - previous_action
            else:
                delta_vec = np.zeros(6)
            
            # Swap first and second elements of delta_vec
            temp = delta_vec[0]
            delta_vec[0] = delta_vec[1]
            delta_vec[1] = temp
            delta_vec_norm = delta_vec / np.linalg.norm(delta_vec) if np.linalg.norm(delta_vec) != 0 else np.zeros(6)

            # Impulse action disturbance every 50 steps
            if action_impulse_level > 0 and step % 50 == 0 and step > 50:
                action += action_impulse_level * delta_vec_norm * action_sigma * np.random.randn(6)
                action = np.clip(action, self.low, self.high)
            
            # Bias action disturbance (fixed offset after bias_start_step)
            if action_bias_level > 0 and step >= bias_start_step:
                action += bias_offset
                action = np.clip(action, self.low, self.high)
            
            # Sine disturbance every step
            if action_sine_level > 0:
                sine_value = np.sin(2 * np.pi * step / 50)
                action += action_sine_level * action_sigma * sine_value
                action = np.clip(action, self.low, self.high)
            
            # Run sub-steps with same action and target (no disturbances in pre-run)
            for hold_number in range(num_substeps):
                next_state, _, _, _, _, _, _, _, _ = self.env.step(action, target)

            actual_positions.append(next_state)
            target_positions.append(target)
            
            # White feedback noise every step
            if feedback_noise_level > 0:
                next_state[:3] += feedback_noise_level * pos_sigma * np.random.randn(3)

            # Update achieved_pos for next iteration
            achieved_pos = next_state
            
            # Update previous_action (clean action before disturbances)
            previous_action = self.agent.act(input_state, noise_level=0.0)
        
        actual_positions = np.array(actual_positions)
        target_positions = np.array(target_positions)
        
        return actual_positions, target_positions

    def run_hybrid(self, file_path, feedback_noise_level=0.0, action_impulse_level=0.0, 
                   action_bias_level=0.0, bias_start_step=120, action_sine_level=0.0, 
                   cont_dt=0.1, sim_dt=0.01, velocity=1.0):
        np.random.seed(self.seed)
        targets, num_substeps, pos_sigma, action_sigma = self._run_common_setup(file_path, cont_dt, sim_dt, velocity)
        
        # Reset for main simulation
        achieved_pos = self.env.reset(new_ep=1)
        achieved_pos = np.array([0.0, 0.0, self.env.observation_space.high[0]])
        action = np.zeros(6, dtype=np.float32)
        bias_offset = action_bias_level * np.random.uniform(low=self.low, high=self.high, size=action.shape)
        
        # Lists to record
        actual_positions = []
        target_positions = []
        
        self.pid_hybrid.reset()
        previous_action = None
        
        # Main simulation loop
        for step in range(len(targets)):
            target = targets[step]
            
            # Compute error
            error = target - achieved_pos
            
            # PID computes full 3D force directly from error (not delta)
            adj_pos = self.pid_hybrid.compute(error)  # [Fx, Fy, Fz] based on error
            # Sum with desired position for actor input
            adjusted_pos = target + adj_pos
            # Actor input is just the position (3D)
            input_state = adjusted_pos
            
            # Get action from agent (deterministic, noise added separately)
            action = self.agent.act(input_state, noise_level=0.0)  # No noise here
            
            # Clip to bounds
            action = np.clip(action, self.low, self.high)
            
            # Compute delta_vec for impulse
            if step > 0:
                delta_vec = action - previous_action
            else:
                delta_vec = np.zeros(6)
            
            # Swap first and second elements of delta_vec
            temp = delta_vec[0]
            delta_vec[0] = delta_vec[1]
            delta_vec[1] = temp
            delta_vec_norm = delta_vec / np.linalg.norm(delta_vec) if np.linalg.norm(delta_vec) != 0 else np.zeros(6)

            # Impulse action disturbance every 50 steps
            if action_impulse_level > 0 and step % 50 == 0 and step > 50:
                action += action_impulse_level * delta_vec_norm * action_sigma * np.random.randn(6)
                action = np.clip(action, self.low, self.high)
            
            # Bias action disturbance (fixed offset after bias_start_step)
            if action_bias_level > 0 and step >= bias_start_step:
                action += bias_offset
                action = np.clip(action, self.low, self.high)
            
            # Sine disturbance every step
            if action_sine_level > 0:
                sine_value = np.sin(2 * np.pi * step / 50)
                action += action_sine_level * action_sigma * sine_value
                action = np.clip(action, self.low, self.high)
            
            # Run sub-steps with same action and target (no disturbances in pre-run)
            for hold_number in range(num_substeps):
                next_state, _, _, _, _, _, _, _, _ = self.env.step(action, target)

            actual_positions.append(next_state)
            target_positions.append(target)
            
            # White feedback noise every step
            if feedback_noise_level > 0:
                next_state[:3] += feedback_noise_level * pos_sigma * np.random.randn(3)

            # Update achieved_pos for next iteration
            achieved_pos = next_state
            
            # Update previous_action (clean action before disturbances)
            previous_action = self.agent.act(input_state, noise_level=0.0)
        
        actual_positions = np.array(actual_positions)
        target_positions = np.array(target_positions)
        
        return actual_positions, target_positions

    def run_pid(self, file_path, feedback_noise_level=0.0, action_impulse_level=0.0, 
                action_bias_level=0.0, bias_start_step=120, action_sine_level=0.0, 
                cont_dt=0.1, sim_dt=0.01, velocity=1.0):
        np.random.seed(self.seed)
        targets, num_substeps, pos_sigma, action_sigma = self._run_common_setup(file_path, cont_dt, sim_dt, velocity)
        
        # Reset for main simulation
        achieved_pos = self.env.reset(new_ep=1)
        achieved_pos = np.array([0.0, 0.0, self.env.observation_space.high[0]])
        action = np.zeros(6, dtype=np.float32)
        bias_offset = action_bias_level * np.random.uniform(low=self.low, high=self.high, size=action.shape)
        
        # Lists to record
        actual_positions = []
        target_positions = []

        self.pid_standalone.reset()
        previous_action = None

        # Simulation loop
        for step in range(len(targets)):
            target = targets[step]

            # Compute error
            error = target - achieved_pos

            # PID computes full 6D wrench directly from error (MODIFIED)
            action = self.pid_standalone.compute(error, self.env.action_space.high, self.env.action_space.low)  # [Fx, Fy, Fz, Tx, Ty, Tz] based on error
            
            # Compute delta_vec for impulse
            if step > 0:
                delta_vec = action - previous_action
            else:
                delta_vec = np.zeros(6)
            
            # Swap first and second elements of delta_vec
            temp = delta_vec[0]
            delta_vec[0] = delta_vec[1]
            delta_vec[1] = temp
            delta_vec_norm = delta_vec / np.linalg.norm(delta_vec) if np.linalg.norm(delta_vec) != 0 else np.zeros(6)

            # Impulse action disturbance every 50 steps
            if action_impulse_level > 0 and step % 50 == 0 and step > 50:
                action += action_impulse_level * delta_vec_norm * action_sigma * np.random.randn(6)
                action = np.clip(action, self.low, self.high)

            # Bias action disturbance (fixed offset after bias_start_step)
            if action_bias_level > 0 and step >= bias_start_step:
                action += bias_offset
                action = np.clip(action, self.low, self.high)
            
            # Sine disturbance every step
            if action_sine_level > 0:
                sine_value = np.sin(2 * np.pi * step / 50)
                action += action_sine_level * action_sigma * sine_value

            # Run sub-steps with same action and target (no disturbances in pre-run)
            for hold_number in range(num_substeps):
                next_state, _, _, _, _, _, _, _, _ = self.env.step(action, target)

            actual_positions.append(next_state)
            target_positions.append(target)

            # White feedback noise every step
            if feedback_noise_level > 0:
                next_state[:3] += feedback_noise_level * pos_sigma * np.random.randn(3)

            # Update achieved_pos for next iteration
            achieved_pos = next_state
            
            # Update previous_action (clean action before disturbances)
            previous_action = self.pid_standalone.compute(error, self.env.action_space.high, self.env.action_space.low)

        actual_positions = np.array(actual_positions)
        target_positions = np.array(target_positions)
        
        return actual_positions, target_positions
